<a href="https://colab.research.google.com/github/sanjaliroy/berkeley-homes-wildfire-agent-simulation/blob/main/wildfire_transcript_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homeowners Interview Transcript Pre-processing

Course: INFO 290: Fundamentals of Generative AI (Spring 2026)







Creator: Amrita Nambiar

What this notebook covers:
1.   Transcription Cleaning: Filters raw text to isolate homeowner responses.
2.   AI-Driven Extraction: Uses Gemini 2.5 Flash model to transform cleaned transcripts into structured profiles.
3.  Data Export: Saves results to Google Drive as agents_extracted.yaml (profiles) and extraction_debug.jsonl (logs).

Why this matters:


1.   Agent Persona Creation: The extracted profiles form the basis for creating realistic and diverse agent personas in a wildfire mitigation simulation.
2.    Structured Data: Converts unstructured interview data into a structured format that can be easily consumed by simulation models.
3.    Automated Extraction: Leverages Generative AI to automate the extraction process, reducing manual effort and ensuring consistency.



## Install Dependencies

In [ ]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.1 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata, drive
import google.genai as genai
from pathlib import Path

import re
import json
import yaml
import google.generativeai as genai
from docx import Document
from pathlib import Path
import pandas as pd

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Configuration

In [ ]:
# Google Drive folder with interview transcripts
transcripts_folder_path = "/content/drive/MyDrive/INFO 290: Intro to Gen AI/Interview Transcripts"

# Output folder
output_folder_path = "/content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs"

# Output filenames
agents_yaml_filename = "agents_extracted.yaml"
debug_jsonl_filename = "extraction_debug.jsonl"

# Model for extraction
model = "models/gemini-2.5-flash"
max_tokens = 2000

# Interviewer name to filter out from transcripts
interviewer_name = "Amrita"

print("Configuration completed")
print(f"Transcripts folder : {transcripts_folder_path}")
print(f"Output folder      : {output_folder_path}")
print(f"Model              : {model}")

Configuration completed
Transcripts folder : /content/drive/MyDrive/INFO 290: Intro to Gen AI/Interview Transcripts
Output folder      : /content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs
Model              : models/gemini-2.5-flash


## Mount Google Drive & Load API Key

In [ ]:
# Mount Google Drive
drive.mount("/content/drive")
print("Google Drive mounted")

# Load API key
wildfire_api_key = userdata.get('wildfire')

if wildfire_api_key:
    # Set it as GOOGLE_API_KEY environment variable
    os.environ["GOOGLE_API_KEY"] = wildfire_api_key
    genai.configure(api_key=wildfire_api_key)
    print("API key loaded.")
else:
    print("API key not found.")

# Verify transcript folder exists
transcript_dir = Path(transcripts_folder_path)
if not transcript_dir.exists():
    print(f"\nTranscript folder not found: {transcripts_folder_path}")
else:
    docx_files = sorted(transcript_dir.glob("*.docx"))
    print(f"\nFound {len(docx_files)} .docx transcript(s):")
    for f in docx_files:
        print(f"   • {f.name}")

Mounted at /content/drive
Google Drive mounted
API key loaded.

Found 11 .docx transcript(s):
   • Beth - Mar 6 at 7_03 PM.docx
   • Charles - Mar 6 at 8_06 PM.docx
   • Edward - Mar 11 at 2_06 PM.docx
   • Isabelle G - Feb 02 2026, 5_00 PM.docx
   • Jennifer - Feb 25 at 1_03 PM.docx
   • Lola - Feb 23 at 5_09 PM.docx
   • Michel_.docx
   • Ross - Feb 25 at 10_21 AM.docx
   • Steven L - Feb 02 2026, 12_00 PM.docx
   • Susan_.docx
   • Yen_.docx


## Helper Functions

In [ ]:
# Extract plain text from .docx
def extract_text_from_docx(docx_path: Path) -> str:
    # Read all paragraphs from a .docx file and return as plain text
    doc = Document(str(docx_path))
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    return "\n\n".join(paragraphs)

In [ ]:
# Filter out interviewer turns
def filter_to_interviewee_only(transcript: str, interviewer_name: str = "Amrita") -> str:
    """
    For timestamped transcripts,strip out all interviewer turns. Return homeowner speech only.
    For unformatted transcripts, return full text unchanged.
    """
    # Detect if transcript uses speaker-timestamp format
    if not re.search(r'\w[\w\s]+:\s+\d{2}:\d{2}', transcript):
        return transcript  # if unformatted, return as it is

    lines = transcript.split('\n')
    filtered = []
    include = True

    for line in lines:
        speaker_match = re.match(r'^([\w][\w\s]+):\s+\d{2}:\d{2}', line)
        if speaker_match:
            speaker = speaker_match.group(1).strip()
            include = interviewer_name not in speaker
        if include:
            filtered.append(line)

    return '\n'.join(filtered)


In [ ]:
# Extraction prompt
extraction_prompt = """\
You are a research assistant helping build realistic agent personas for a wildfire \
mitigation simulation set in the Berkeley Hills, California.

Below is an interview transcript with a Berkeley Hills homeowner. Your job is to \
extract a structured profile that will be used to initialise a simulation agent.

TRANSCRIPT:
{transcript}

Extract the following and respond ONLY with valid JSON (no markdown, no preamble, \
no trailing text):

{{
  "agent_id": "<firstname_lowercase — e.g. ross, yen, beth>",
  "display_name": "<First name only — e.g. Ross>",
  "archetype": "<The Resourceful DIYer | The Engaged Community Member | The Skeptical Homeowner | The Compliant Citizen | The Overwhelmed Resident>",
  "age": "<age or age range — e.g. 35>",
  "location": "<Berkeley Hills (Berkeley Woods Firewise area) | specific neighborhood or general area>",
  "persona": "<3-5 sentence paragraph in third person capturing: who they are, \
how long they have lived there, their attitude toward fire risk, their relationship \
with the Ember ordinance and defensible space requirements, their trust in government \
and fire institutions, and any distinctive personality traits revealed in the interview>",
  "risk_zone": "<high | medium | low — infer from location: ridge/hilltop=high, \
mid-hill=medium, flatter/lower=low>",
  "tilden_proximity": "<front_line | mid_hills | moderate_distance | far | unknown>",
  "property_type": "<single_family | rental | residential | unknown>",
  "lot_size": "<large | standard | small | unknown>",
  "roof_type": "<tile | composition_shingle | metal | clay | slate | concrete | wood | asphalt | unknown>",
  "siding": "<stucco | wood | fiber_cement | unknown>",
  "eaves_vents": "<addressed | not_addressed | unknown>",
  "vegetation_zone0": "<fully_compliant | partial | non_compliant | unknown>",
  "vegetation_zone1": "<fully_compliant | partial | non_compliant | unknown>",
  "fence_gates": "<metal | wood | unknown>",
  "institutional_trust": "<high | medium | low — based on how they talk about the \
fire department, county, inspectors, and the Ember ordinance>",
  "compliance_status": "<compliant | in_progress | non_compliant | partial | unknown>",
  "insurance_status": "<retained | non_renewed | dropped | unknown>",
  "years_at_property": <integer or null>,
  "firewise_group": "<name of firewise group or null>",
  "firewise_role": "<active_member | passive_member | leader | unknown | null>",
  "spending_so_far": "<minimal | moderate | high | unknown>",
  "anticipated_costs": "<minimal | moderate | high | unknown>",
  "financial_flexibility": "<high | moderate | low | unknown>",
  "social_network": "<active | limited | inactive | unknown>",
  "action_driver": "<list down what initiatives could potentially help them to take wildfire mitigation actions, it can be \
  grants,resources, discounts, trusted contractors,better guideance, etc, or mark unknown if it wasn't discussed>",
  "key_motivation": "<list down what motivates them to take wildfire mitigation actions, it can be \
  to demonstrate compliance can look good, avoid_penalties, protect their property, avoid losing their insurance, \
  social_awareness, etc or mark unknown if it wasn't discussed>",
  "diy_orientation": "<high | medium | low | unknown>",
  "information_seeking": "<obsessive | moderate | limited | unknown>",
  "emotional_style": "<creative_and_determined | anxious_and_frustrated | resigned_but_compliant | skeptical_and_resistant | unknown>",
  "communication_preferences": "<official_mail | news_media | social | \
direct_experience — pick the single channel type that best matches how they say \
they receive and trust information>",
  "seed_narrative": "<A 3-5 sentence, multi-line paragraph that captures the agent's core story, motivations, and current situation regarding wildfire mitigation. Refer to the example provided by the user.>",
  "core_beliefs": [
    "<belief 1>",
    "<belief 2>",
    "<belief 3>"
  ],
  "memory_seeds": [
    {{
      "description": "<specific first-person memory grounded in the transcript — past tense, \
concrete detail, e.g. 'I built a brick patio using salvaged bricks'>",
      "importance": <integer 1-10>,
      "type": "<observation | reflection | conversation>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10>,
      "type": "<observation | reflection | conversation>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10>,
      "type": "<observation | reflection | conversation>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10>,
      "type": "<observation | reflection | conversation>"
    }},
    {{
      "description": "<another memory>",
      "importance": <integer 1-10>,
      "type": "<observation | reflection | conversation>"
    }}
  ],
  "held_out_responses": {{
    "insurance_non_renewal": {{
      "real_response": "<Concise response if faced with insurance non-renewal>",
      "context": "<Brief context for this response>"
    }},
    "official_regulatory_notice": {{
      "real_response": "<Concise response if faced with an official regulatory notice>",
      "context": "<Brief context for this response>"
    }}
  }},
  "key_concerns": ["<concern 1>", "<concern 2>", "<concern 3>"],
  "notable_quotes": [
    "<short verbatim quote that captures their voice — under 20 words>",
    "<another quote>"
  ]
}}

Rules:
- memory_seeds MUST be exactly 5 items, specific to THIS person, not generic
- persona must reflect actual personality from the interview, not a generic homeowner
- Do NOT invent facts not supported by the transcript
- If a field is unknown, use null
- Respond with JSON only — no explanation, no markdown fences
"""

In [ ]:
# Call Gemini to extract profile
def extract_agent_profile(model_client: genai.GenerativeModel, transcript: str, name: str) -> tuple:
    # Send transcript to Gemini and return (parsed_dict, raw_text)
    # Truncate very long transcripts (~6000 words) to stay within context budget
    words = transcript.split()
    if len(words) > 6000:
        transcript = ' '.join(words[:6000]) + "\n\n[transcript truncated for length]"

    prompt = extraction_prompt.format(transcript=transcript)

    response = model_client.generate_content(prompt)

    raw_text = response.text.strip()

    # Strip markdown fences if the model adds them despite instructions
    raw_text = re.sub(r'^```(?:json)?\s*', '', raw_text)
    raw_text = re.sub(r'\s*```$', '', raw_text).strip()

    try:
        profile = json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"JSON parse error for {name}: {e}")
        profile = {"agent_id": name.lower(), "parse_error": str(e), "raw": raw_text}

    return profile, raw_text

In [ ]:
# Convert profile to agents.yaml format
def profile_to_agent_entry(profile: dict) -> dict:
    # Reshape extracted profile into the agents.yaml schema
    if "parse_error" in profile:
        return {"id": profile.get("agent_id", "unknown"), "error": profile["parse_error"]}

    return {
        "id":                      profile.get("agent_id", "unknown"),
        "display_name":            profile.get("display_name", "Unknown"),
        "archetype":               profile.get("archetype"),
        "age":                     profile.get("age"),
        "location":                profile.get("location"),
        "persona":                 profile.get("persona", ""),
        "risk_zone":               profile.get("risk_zone", "unknown"),
        "tilden_proximity":        profile.get("tilden_proximity"),
        "property_type":           profile.get("property_type", "unknown"),
        "lot_size":                profile.get("lot_size"),
        "roof_type":               profile.get("roof_type"),
        "siding":                  profile.get("siding"),
        "eaves_vents":             profile.get("eaves_vents"),
        "vegetation_zone0":        profile.get("vegetation_zone0"),
        "vegetation_zone1":        profile.get("vegetation_zone1"),
        "fence_gates":             profile.get("fence_gates"),
        "institutional_trust":     profile.get("institutional_trust", "medium"),
        "compliance_status":       profile.get("compliance_status", "unknown"),
        "insurance_status":        profile.get("insurance_status"),
        "years_at_property":       profile.get("years_at_property"),
        "firewise_group":          profile.get("firewise_group"),
        "firewise_role":           profile.get("firewise_role"),
        "spending_so_far":         profile.get("spending_so_far"),
        "anticipated_costs":       profile.get("anticipated_costs"),
        "financial_flexibility":   profile.get("financial_flexibility"),
        "social_network":          profile.get("social_network"),
        "action_driver":           profile.get("action_driver"),
        "key_motivation":          profile.get("key_motivation"),
        "diy_orientation":         profile.get("diy_orientation"),
        "information_seeking":     profile.get("information_seeking"),
        "emotional_style":         profile.get("emotional_style"),
        "communication_preferences": profile.get("communication_preferences", "social"),
        "seed_narrative":          profile.get("seed_narrative", ""),
        "core_beliefs":            profile.get("core_beliefs", []),
        "memory_seeds":            profile.get("memory_seeds", []),
        "held_out_responses":      profile.get("held_out_responses", {}),
        "key_concerns":            profile.get("key_concerns", []),
        "notable_quotes":          profile.get("notable_quotes", []),
    }


print("Helper functions defined")

Helper functions defined


## Run Extraction (All Transcripts)

In [ ]:
# Setup
gemini_model_client = genai.GenerativeModel(model) # Initialize Gemini model

transcript_dir = Path(transcripts_folder_path)
output_dir = Path(output_folder_path)
output_dir.mkdir(parents=True, exist_ok=True)

docx_files = sorted(transcript_dir.glob("*.docx"))
print(f"Processing {len(docx_files)} transcripts...\n")
print("=" * 60)

agents = []
debug_entries = []
errors = []

# Main extraction loop
for i, docx_path in enumerate(docx_files, 1):
    # Derive a short name from the filename
    stem = docx_path.stem
    name = stem.split(" - ")[0].split("_")[0].strip()

    print(f"[{i}/{len(docx_files)}] {name}  ({docx_path.name})")

    try:
        # Extract raw text
        full_text = extract_text_from_docx(docx_path)
        word_count = len(full_text.split())
        print(f"         {word_count:,} words extracted")

        # Filter to homeowner voice only
        homeowner_text = filter_to_interviewee_only(full_text, interviewer_name)
        filtered_wc = len(homeowner_text.split())
        if filtered_wc < word_count:
            print(f"         → filtered to {filtered_wc:,} words (interviewer turns removed)")

        # Call Gemini
        profile, raw_response = extract_agent_profile(gemini_model_client, homeowner_text, name)

        # Convert to agents.yaml format
        agent_entry = profile_to_agent_entry(profile)
        agents.append(agent_entry)

        # Save debug info
        debug_entries.append({
            "file": docx_path.name,
            "name": name,
            "word_count_full": word_count,
            "word_count_filtered": filtered_wc,
            "raw_llm_response": raw_response,
            "parsed_profile": profile
        })

        # Print summary row
        print(f"         agent_id={agent_entry.get('id')} | "
              f"risk={agent_entry.get('risk_zone')} | "
              f"trust={agent_entry.get('institutional_trust')} | "
              f"compliance={agent_entry.get('compliance_status')}")

    except Exception as e:
        print(f"         ERROR: {e}")
        errors.append({"file": docx_path.name, "error": str(e)})

    print()

print("=" * 60)
print(f"Done. {len(agents)} agents extracted, {len(errors)} errors.")

Processing 11 transcripts...

[1/11] Beth  (Beth - Mar 6 at 7_03 PM.docx)
         7,636 words extracted
         agent_id=beth | risk=high | trust=low | compliance=in_progress

[2/11] Charles  (Charles - Mar 6 at 8_06 PM.docx)
         6,234 words extracted
         agent_id=preston_rd_homeowner | risk=high | trust=medium | compliance=in_progress

[3/11] Edward  (Edward - Mar 11 at 2_06 PM.docx)
         1,193 words extracted
         agent_id=homeowner | risk=high | trust=low | compliance=compliant

[4/11] Isabelle G  (Isabelle G - Feb 02 2026, 5_00 PM.docx)
         6,935 words extracted
         → filtered to 4,390 words (interviewer turns removed)
         agent_id=isabelle | risk=high | trust=high | compliance=compliant

[5/11] Jennifer  (Jennifer - Feb 25 at 1_03 PM.docx)
         4,350 words extracted
         agent_id=jennifer | risk=high | trust=medium | compliance=in_progress

[6/11] Lola  (Lola - Feb 23 at 5_09 PM.docx)
         2,538 words extracted
         agent_id=jane 

## Save Outputs to Google Drive

In [ ]:
# Save agents_extracted.yaml
agents_yaml_path = output_dir / agents_yaml_filename
with open(agents_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(
        {"agents": agents},
        f,
        default_flow_style=False,
        allow_unicode=True,
        sort_keys=False
    )

print(f"Saved {len(agents)} agents → {agents_yaml_path}")

# Save extraction_debug.jsonl
debug_jsonl_path = output_dir / debug_jsonl_filename

with open(debug_jsonl_path, "w", encoding="utf-8") as f:
    for entry in debug_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Saved debug log    → {debug_jsonl_path}")

if errors:
    print(f"\n{len(errors)} file(s) failed:")
    for e in errors:
        print(f"   • {e['file']}: {e['error']}")

Saved 11 agents → /content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs/agents_extracted.yaml
Saved debug log    → /content/drive/MyDrive/INFO 290: Intro to Gen AI/outputs/extraction_debug.jsonl


Summary Table

In [ ]:
import pandas as pd

rows = []
for a in agents:
    rows.append({
        "ID":                      a.get("id", "?"),
        "Agent":                   a.get("display_name", "?"),
        "Archetype":               a.get("archetype", "?"),
        "Age":                     a.get("age", "?"),
        "Location":                a.get("location", "?"),
        "Persona Present":         "Yes" if a.get("persona") else "No",
        "Risk Zone":               a.get("risk_zone", "?"),
        "Tilden Proximity":        a.get("tilden_proximity", "?"),
        "Property Type":           a.get("property_type", "?"),
        "Lot Size":                a.get("lot_size", "?"),
        "Roof Type":               a.get("roof_type", "?"),
        "Siding":                  a.get("siding", "?"),
        "Eaves Vents":             a.get("eaves_vents", "?"),
        "Veg Zone 0":              a.get("vegetation_zone0", "?"),
        "Veg Zone 1":              a.get("vegetation_zone1", "?"),
        "Fence Gates":             a.get("fence_gates", "?"),
        "Trust":                   a.get("institutional_trust", "?"),
        "Compliance":              a.get("compliance_status", "?"),
        "Insurance Status":        a.get("insurance_status", "?"),
        "Years at Property":       a.get("years_at_property", "?"),
        "Firewise Group":          a.get("firewise_group", "?"),
        "Firewise Role":           a.get("firewise_role", "?"),
        "Spending So Far":         a.get("spending_so_far", "?"),
        "Anticipated Costs":       a.get("anticipated_costs", "?"),
        "Financial Flexibility":   a.get("financial_flexibility", "?"),
        "Social Network":          a.get("social_network", "?"),
        "Action Driver":           a.get("action_driver", "?"),
        "Key Motivation":          a.get("key_motivation", "?"),
        "DIY Orientation":         a.get("diy_orientation", "?"),
        "Information Seeking":     a.get("information_seeking", "?"),
        "Emotional Style":         a.get("emotional_style", "?"),
        "Comms Channel":           a.get("communication_preferences", "?"),
        "Narrative Present":       "Yes" if a.get("seed_narrative") else "No",
        "Core Beliefs Count":      len(a.get("core_beliefs", [])),
        "Memory Seeds Count":      len(a.get("memory_seeds", [])),
        "Held Out Responses Present": "Yes" if a.get("held_out_responses", {}) else "No",
        "Key Concerns Count":      len(a.get("key_concerns", [])),
        "Notable Quotes Count":    len(a.get("notable_quotes", [])),
    })

df = pd.DataFrame(rows)
print("\nEXTRACTED AGENT SUMMARY")
display(df)

# Distribution checks — useful for ensuring variance across agent pool
print("\nDISTRIBUTIONS")
for col in ["ID", "Archetype", "Age", "Location", "Persona Present", "Risk Zone", "Tilden Proximity",
            "Property Type", "Lot Size", "Roof Type", "Siding", "Eaves Vents", "Veg Zone 0",
            "Veg Zone 1", "Fence Gates", "Trust", "Compliance", "Insurance Status", "Years at Property",
            "Firewise Group", "Firewise Role", "Spending So Far", "Anticipated Costs", "Financial Flexibility",
            "Social Network", "Action Driver", "Key Motivation", "DIY Orientation", "Information Seeking",
            "Emotional Style", "Comms Channel", "Narrative Present", "Core Beliefs Count",
            "Memory Seeds Count", "Held Out Responses Present", "Key Concerns Count", "Notable Quotes Count"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().to_string())



EXTRACTED AGENT SUMMARY


,ID,Agent,Archetype,Age,Location,Persona Present,Risk Zone,Tilden Proximity,Property Type,Lot Size,...,DIY Orientation,Information Seeking,Emotional Style,Comms Channel,Narrative Present,Core Beliefs Count,Memory Seeds Count,Held Out Responses Present,Key Concerns Count,Notable Quotes Count
0,beth,Beth,The Skeptical Homeowner,65+,"Berkeley Hills (top of the hills, near Grizzly...",Yes,high,front_line,single_family,standard,...,medium,limited,skeptical_and_resistant,social,Yes,4,5,Yes,4,2
1,preston_rd_homeowner,Preston Road Resident,The Engaged Community Member,None,"Preston Road, top of the hill, Berkeley Hills",Yes,high,front_line,single_family,standard,...,low,obsessive,anxious_and_frustrated,official_mail,Yes,3,5,Yes,3,2
2,homeowner,Homeowner,The Skeptical Homeowner,65+,Berkeley Hills (east side of the Hills),Yes,high,front_line,single_family,large,...,high,moderate,skeptical_and_resistant,direct_experience,Yes,4,5,Yes,4,3
3,isabelle,Isabelle,The Engaged Community Member,None,Berkeley Hills (Berkeley Woods Firewise area),Yes,high,front_line,single_family,standard,...,medium,obsessive,creative_and_determined,direct_experience,Yes,4,5,Yes,4,3
4,jennifer,Jennifer,The Resourceful DIYer,None,Berkeley Hills (Berkeley Woods Firewise area),Yes,high,unknown,single_family,unknown,...,high,moderate,creative_and_determined,news_media,Yes,3,5,Yes,3,2
5,jane,Jane,The Resourceful DIYer,60s+,Berkeley Hills (Berkeley Woods Firewise area),Yes,high,front_line,single_family,None,...,high,obsessive,anxious_and_frustrated,direct_experience,Yes,5,5,Yes,5,3
6,marc,Marc,The Engaged Community Member,60s,Berkeley Hills (Creston Ridge),Yes,high,front_line,single_family,standard,...,high,obsessive,creative_and_determined,direct_experience,Yes,3,5,Yes,4,4
7,jordan,Jordan,The Compliant Citizen,None,Berkeley Hills (Berkeley Woods Firewise area),Yes,high,front_line,single_family,None,...,high,moderate,anxious_and_frustrated,direct_experience,Yes,4,5,Yes,4,3
8,steven,Steven,The Compliant Citizen,None,Berkeley Hills,Yes,high,mid_hills,single_family,None,...,unknown,moderate,confident_and_opinionated,official_mail,Yes,4,5,Yes,4,4
9,susan,Susan,The Engaged Community Member,60-75,Northeast Berkeley Hills,Yes,high,front_line,single_family,None,...,unknown,moderate,creative_and_determined,direct_experience,Yes,5,5,Yes,5,5



DISTRIBUTIONS

ID:
ID
beth                    1
preston_rd_homeowner    1
homeowner               1
isabelle                1
jennifer                1
jane                    1
marc                    1
jordan                  1
steven                  1
susan                   1
yen                     1

Archetype:
Archetype
The Engaged Community Member    5
The Skeptical Homeowner         2
The Resourceful DIYer           2
The Compliant Citizen           2

Age:
Age
65+      2
60s+     1
60s      1
60-75    1
50s      1

Location:
Location
Berkeley Hills (Berkeley Woods Firewise area)           5
Preston Road, top of the hill, Berkeley Hills           1
Berkeley Hills (top of the hills, near Grizzly Peak)    1
Berkeley Hills (east side of the Hills)                 1
Berkeley Hills (Creston Ridge)                          1
Berkeley Hills                                          1
Northeast Berkeley Hills                                1

Persona Present:
Persona Present
Yes    1

##Inspect Individual Agent Profile

In [ ]:
# Change this to the agent_id you want to inspect
inspect_agent = "jennifer"

match = next((a for a in agents if a.get("id") == inspect_agent), None)

if match is None:
    print(f"Agent '{inspect_agent}' not found. Available: {[a.get('id') for a in agents]}")
else:
    print(f"=== {match['display_name'].upper()} ===")
    print(f"\nPERSONA:")
    print(f"  {match['persona']}")

    print(f"\nKEY ATTRIBUTES:")
    print(f"  archetype            : {match.get('archetype')}")
    print(f"  age                  : {match.get('age')}")
    print(f"  location             : {match.get('location')}")
    print(f"  risk_zone            : {match.get('risk_zone')}")
    print(f"  tilden_proximity     : {match.get('tilden_proximity')}")
    print(f"  property_type        : {match.get('property_type')}")
    print(f"  lot_size             : {match.get('lot_size')}")
    print(f"  roof_type            : {match.get('roof_type')}")
    print(f"  siding               : {match.get('siding')}")
    print(f"  eaves_vents          : {match.get('eaves_vents')}")
    print(f"  vegetation_zone0     : {match.get('vegetation_zone0')}")
    print(f"  vegetation_zone1     : {match.get('vegetation_zone1')}")
    print(f"  fence_gates          : {match.get('fence_gates')}")
    print(f"  institutional_trust  : {match.get('institutional_trust')}")
    print(f"  compliance_status    : {match.get('compliance_status')}")
    print(f"  insurance_status     : {match.get('insurance_status')}")
    print(f"  years_at_property    : {match.get('years_at_property')}")
    print(f"  firewise_group       : {match.get('firewise_group')}")
    print(f"  firewise_role        : {match.get('firewise_role')}")
    print(f"  spending_so_far      : {match.get('spending_so_far')}")
    print(f"  anticipated_costs    : {match.get('anticipated_costs')}")
    print(f"  financial_flexibility: {match.get('financial_flexibility')}")
    print(f"  social_network       : {match.get('social_network')}")
    print(f"  action_driver        : {match.get('action_driver')}")
    print(f"  key_motivation       : {match.get('key_motivation')}")
    print(f"  diy_orientation      : {match.get('diy_orientation')}")
    print(f"  information_seeking  : {match.get('information_seeking')}")
    print(f"  emotional_style      : {match.get('emotional_style')}")
    print(f"  communication_preferences: {match.get('communication_preferences')}")

    print(f"\nSEED MEMORIES:")
    # memory_seeds is the correct key, not seed_memories
    for i, mem in enumerate(match.get('memory_seeds', []), 1):
        print(f"  {i}. {mem.get('description', 'N/A')}")

    print(f"\nKEY CONCERNS:")
    for concern in match.get('key_concerns', []):
        print(f"  • {concern}")

    print(f"\nNOTABLE QUOTES:")
    for quote in match.get('notable_quotes', []):
        print(f'  "{quote}"')


=== JENNIFER ===

PERSONA:
  Jennifer is a Berkeley Hills homeowner who, along with her 'handy husband,' actively works on making her property fire-safe. She is resourceful, undertaking DIY projects like building a patio for Zone Zero, and seeks out practical solutions. She is keenly aware of the financial and aesthetic challenges of compliance, and expresses frustration with the capricious nature of insurance and the lack of accessible, specific guidance for hill properties. She values community involvement and practical solutions over abstract regulations, trusting her own research and local connections.

KEY ATTRIBUTES:
  archetype            : The Resourceful DIYer
  age                  : None
  location             : Berkeley Hills (Berkeley Woods Firewise area)
  risk_zone            : high
  tilden_proximity     : unknown
  property_type        : single_family
  lot_size             : unknown
  roof_type            : unknown
  siding               : unknown
  eaves_vents       

## Preview agents.yaml Output

In [ ]:
# Print the first 2 agents from the YAML to verify formatting
preview = {"agents": agents[:2]}
print(yaml.dump(preview, default_flow_style=False, allow_unicode=True, sort_keys=False))

agents:
- id: beth
  display_name: Beth
  archetype: The Skeptical Homeowner
  age: 65+
  location: Berkeley Hills (top of the hills, near Grizzly Peak)
  persona: Beth is a long-time Berkeley Hills resident and a working lawyer who deeply
    values her established garden. She is pragmatic and compliant with wildfire mitigation
    efforts on her property, having already spent significant time and money. However,
    she harbors strong skepticism and frustration regarding the inconsistent guidance
    from fire officials, the perceived inexperience of inspectors, and the city's
    inaction on larger fire breaks, which she deems "ridiculous." Despite her criticism,
    she actively seeks out trusted contractors via her social network and is determined
    to protect her home.
  risk_zone: high
  tilden_proximity: front_line
  property_type: single_family
  lot_size: standard
  roof_type: null
  siding: null
  eaves_vents: addressed
  vegetation_zone0: partial
  vegetation_zone1: parti

## Re-run a Single Agent

Use this if one extraction failed or produced a poor result. It re-runs just that one file without re-processing the others.

In [ ]:
# Set to the filename you want to re-run (just the filename, not the full path)
rerun_file = "Ross - Feb 25 at 10_21 AM.docx"

rerun_path = transcript_dir / rerun_file

if not rerun_path.exists():
    print(f"File not found: {rerun_path}")
else:
    name = rerun_file.split(" - ")[0].split("_")[0].strip()
    print(f"Re-running extraction for: {name}")

    full_text = extract_text_from_docx(rerun_path)
    homeowner_text = filter_to_interviewee_only(full_text, interviewer_name)
    profile, raw_response = extract_agent_profile(gemini_model_client, homeowner_text, name)
    agent_entry = profile_to_agent_entry(profile)

    # Replace in agents list if already exists, otherwise append
    existing_ids = [a.get("id") for a in agents]
    if agent_entry.get("id") in existing_ids:
        idx = existing_ids.index(agent_entry.get("id"))
        agents[idx] = agent_entry
        print(f"Replaced existing entry for '{agent_entry.get('id')}'")
    else:
        agents.append(agent_entry)
        print(f"Added new entry for '{agent_entry.get('id')}'")

    print(f"\nResult:")
    print(json.dumps(agent_entry, indent=2))

    # Optionally re-save the YAML
    with open(agents_yaml_path, "w", encoding="utf-8") as f:
        yaml.dump({"agents": agents}, f, default_flow_style=False,
                  allow_unicode=True, sort_keys=False)
    print(f"\nUpdated {agents_yaml_path}")

Re-running extraction for: Ross
Added new entry for 'pat'

Result:
{
  "id": "pat",
  "display_name": "Pat",
  "archetype": "The Compliant Citizen",
  "age": null,
  "location": "Berkeley Hills (Wildcat Canyon Road near Tilden)",
  "persona": "Pat has lived in the Berkeley Hills for five years and is deeply concerned about wildfire risk, believing it's a matter of 'when not if.' They are highly motivated to comply with the Ember ordinance, possess the financial flexibility to hire help, and are willing to put in personal labor. Pat is frustrated by the ambiguity and inconsistent interpretations of the regulations by inspectors, longing for clear, direct guidance from the fire department. They are an active member of their Firewise community and are also concerned about their neighbors' ability to comply.",
  "risk_zone": "high",
  "tilden_proximity": "front_line",
  "property_type": "single_family",
  "lot_size": "unknown",
  "roof_type": null,
  "siding": "wood",
  "eaves_vents": "unk